# 第一天：Agent智能体prompt + skill

## 1-1 普通AI

这个 Notebook 只负责演示和调试。

模型配置和调用逻辑全部来自 `src/`。

运行前请复制 `.env.example` 为 `.env`，并填写自己的 `DEEPSEEK_API_KEY`。

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "teacher":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.artifacts import PROMPT_PATH, read_markdown
from src.facade import invoke
from src.model import get_llm

## 1-2. 检查模型配置

模型配置和调用逻辑全部来自 `src/`。

运行前请复制 `.env.example` 为 `.env`，并填写自己的 `DEEPSEEK_API_KEY`。

In [ ]:
llm = get_llm()
print(f"模型已配置成功✌️：{llm.model_name}")

## 1-3. 运行 普通对话

V0 直接调用模型，还没有添加课程 Prompt、Skill、Knowledge 或 Workflow。

In [ ]:
question = "I ____ (read) this book three times.用什么时态我一直搞不清楚？"
result = invoke("V0", question)
if result["error"]:
    raise RuntimeError(result["error"])
print(result["text"])

## 2-1. 自定义 Prompt

V1 每次运行都会重新读取 `student/prompt.md`。

In [ ]:
print(read_markdown(PROMPT_PATH))

## 2-2. 搭建 🦉苏格拉底AI智能体教练

这里不再使用 V0 的预设题目。

In [ ]:
def run_v1_dialogue():
    history = []
    print("🦉苏格拉底AI智能体教练已来到你身边了。")
    # print("请先输入你想学习的问题，输入 /exit 可以结束对话。")

    while True:
        student_message = input("学生：").strip()
        if student_message.lower() in {"/exit", "exit", "退出"}:
            print("期待下次交流。")
            return history
        if not student_message:
            print("输入不能为空，请重新输入。")
            continue

        turn_result = invoke("V1", student_message, history=history)
        if turn_result["error"]:
            raise RuntimeError(turn_result["error"])

        history.extend([
            {"role": "user", "content": student_message},
            {"role": "assistant", "content": turn_result["text"]},
        ])
        print(f"Carl教练：{turn_result['text']}")

## 2-3. 与 🦉苏格拉底AI智能体教练 对话

In [ ]:
v1_dialogue_history = run_v1_dialogue()

## 2-4. 修改 Prompt 后再次观察

可以在对话过程中使用外部编辑器修改 `student/prompt.md` 并保存。

学生提交下一次回答时，V1 会重新读取 Prompt，并继续携带之前的问答历史。

如果需要从头开始，先输入 `/exit`，再重新运行第 6 部分。

## 2-5. 查看统一结果结构

后续 V1 到 V4 会继续沿用这些字段。

In [ ]:
result